# FoodMkt_R — Solution

**Short name:** `FoodMkt_R`. Worked answers for the food-industry adaptation of Packt Cookbook Ch. 4.

**Disclaimer.** Teaching pipeline only — not a recommendation to buy food-company equity or to award a supplier contract.


## 0. Setup


In [ ]:
library(ggplot2)
library(plyr)
library(reshape2)
library(zoo)
theme_set(theme_minimal())
if (requireNamespace("dplyr", quietly = TRUE)) library(dplyr)
if (requireNamespace("tidyr", quietly = TRUE)) library(tidyr)

## 1. Acquire the food snapshot


In [ ]:
foodviz <- read.csv("data/foodviz.csv", stringsAsFactors = FALSE, check.names = FALSE)
head(foodviz[, 1:6])
dim(foodviz)
names(foodviz)
sort(table(foodviz$Sector), decreasing = TRUE)

## 2. Summarize fields


In [ ]:
summary(foodviz[, 1:6])
# Price          last trade of the food-company share
# P/E            price / earnings
# PEG            P/E / expected growth
# Debt/Equity    leverage (food processors often carry working-capital debt)
# Beta           vol vs market; staples often < 1
# Gross Margin   (sales − COGS) / sales
# Food Cost %    input / COGS intensity (higher at restaurants & protein)
# SSS %          same-store sales, restaurants only

## 3. Clean numerics


In [ ]:
clean_numeric <- function(s) {
  s <- gsub("%|\\$|,|\\)|\\(", "", s)
  as.numeric(s)
}
id_cols <- 1:6
num_cols <- 7:ncol(foodviz)
foodviz <- cbind(foodviz[, id_cols], as.data.frame(lapply(foodviz[, num_cols], clean_numeric)))
names(foodviz) <- make.names(names(foodviz))
str(foodviz)
summary(foodviz$Price)
names(foodviz)

## 4. Explore the price distribution


In [ ]:
hist(foodviz$Price, breaks = 100, main = "Price Distribution (raw)", xlab = "Price")
hist(foodviz$Price[foodviz$Price < 150], breaks = 100,
     main = "Price Distribution (Price < $150)", xlab = "Price")

sector_avg_prices <- aggregate(Price ~ Sector, data = foodviz, FUN = mean)
colnames(sector_avg_prices)[2] <- "Sector_Avg_Price"
ggplot(sector_avg_prices, aes(x = Sector, y = Sector_Avg_Price, fill = Sector)) +
  geom_bar(stat = "identity") +
  theme(axis.text.x = element_text(angle = 40, hjust = 1), legend.position = "none") +
  ggtitle("Sector Avg Price (includes GODIVA)")

## 5. Drill-down and drop GODIVA


In [ ]:
industry_avg_prices <- aggregate(Price ~ Sector + Industry, data = foodviz, FUN = mean)
industry_avg_prices <- industry_avg_prices[order(-industry_avg_prices$Price), ]
industry_chart <- subset(industry_avg_prices, Sector == "Confectionery")
ggplot(industry_chart, aes(x = Industry, y = Price, fill = Industry)) +
  geom_bar(stat = "identity") +
  theme(legend.position = "none", axis.text.x = element_text(angle = 25, hjust = 1)) +
  ggtitle("Confectionery Industries — Avg Price")

company_chart <- subset(foodviz, Industry == "Luxury Chocolate")
ggplot(company_chart, aes(x = Company, y = Price, fill = Company)) +
  geom_bar(stat = "identity") +
  theme(legend.position = "none", axis.text.x = element_text(angle = 40, hjust = 1, size = 8)) +
  ggtitle("Luxury Chocolate — Company Prices")

subset(foodviz, Ticker == "GODIVA")[, c("Ticker", "Company", "Price", "Country")]

foodviz <- subset(foodviz, Ticker != "GODIVA")
sector_avg_prices <- aggregate(Price ~ Sector, data = foodviz, FUN = mean)
ggplot(sector_avg_prices, aes(x = Sector, y = Price, fill = Sector)) +
  geom_bar(stat = "identity") +
  theme(axis.text.x = element_text(angle = 40, hjust = 1), legend.position = "none") +
  ggtitle("Sector Avg Price (GODIVA removed)")

## 6. Relative valuation averages


In [ ]:
val_vars <- intersect(c("Price", "P.E", "PEG", "P.S", "P.B"), names(foodviz))
val_vars

sector_avg <- melt(foodviz, id = "Sector")
sector_avg <- subset(sector_avg, variable %in% val_vars)
sector_avg <- na.omit(sector_avg)
sector_avg$value <- as.numeric(sector_avg$value)
sector_avg <- dcast(sector_avg, Sector ~ variable, mean)
ren <- c(Price = "SAvgPrice", P.E = "SAvgPE", PEG = "SAvgPEG", P.S = "SAvgPS", P.B = "SAvgPB")
for (nm in names(ren)) if (nm %in% names(sector_avg)) {
  names(sector_avg)[names(sector_avg) == nm] <- ren[[nm]]
}
sector_avg

industry_avg <- melt(foodviz, id = c("Sector", "Industry"))
industry_avg <- subset(industry_avg, variable %in% val_vars)
industry_avg <- na.omit(industry_avg)
industry_avg$value <- as.numeric(industry_avg$value)
industry_avg <- dcast(industry_avg, Sector + Industry ~ variable, mean)
renI <- c(Price = "IAvgPrice", P.E = "IAvgPE", PEG = "IAvgPEG", P.S = "IAvgPS", P.B = "IAvgPB")
for (nm in names(renI)) if (nm %in% names(industry_avg)) {
  names(industry_avg)[names(industry_avg) == nm] <- renI[[nm]]
}

n_before <- nrow(foodviz)
foodviz <- merge(foodviz, sector_avg, by = "Sector")
foodviz <- merge(foodviz, industry_avg, by = c("Sector", "Industry"))
c(before = n_before, after = nrow(foodviz))

## 7. Flags and RelValIndex


In [ ]:
foodviz$SPEUnder <- as.integer(foodviz$P.E < foodviz$SAvgPE)
foodviz$SPEGUnder <- as.integer(foodviz$PEG < foodviz$SAvgPEG)
foodviz$SPSUnder <- as.integer(foodviz$P.S < foodviz$SAvgPS)
foodviz$SPBUnder <- as.integer(foodviz$P.B < foodviz$SAvgPB)
foodviz$SPriceUnder <- as.integer(foodviz$Price < foodviz$SAvgPrice)
foodviz$IPEUnder <- as.integer(foodviz$P.E < foodviz$IAvgPE)
foodviz$IPEGUnder <- as.integer(foodviz$PEG < foodviz$IAvgPEG)
foodviz$IPSUnder <- as.integer(foodviz$P.S < foodviz$IAvgPS)
foodviz$IPBUnder <- as.integer(foodviz$P.B < foodviz$IAvgPB)
foodviz$IPriceUnder <- as.integer(foodviz$Price < foodviz$IAvgPrice)

flag_cols <- c("SPEUnder","SPEGUnder","SPSUnder","SPBUnder","SPriceUnder",
               "IPEUnder","IPEGUnder","IPSUnder","IPBUnder","IPriceUnder")
foodviz$RelValIndex <- rowSums(foodviz[flag_cols], na.rm = TRUE)
hist(foodviz$RelValIndex, breaks = -0.5:10.5, main = "Food RelValIndex", xlab = "score")

potentially_undervalued <- subset(foodviz, RelValIndex >= 8)
head(potentially_undervalued[order(-potentially_undervalued$RelValIndex),
                             c("Ticker", "Company", "Sector", "RelValIndex", "Price")], 15)

## 8. Screen + historical prices + MAs


In [ ]:
names(foodviz)

target_stocks <- subset(
  foodviz,
  Price > 20 & Price < 100 &
    Volume > 10000 &
    Country == "USA" &
    EPS..ttm. > 0 &
    EPS.growth.next.year > 0 &
    EPS.growth.next.5.years > 0 &
    Total.Debt.Equity < 1 &
    Beta < 1.5 &
    Institutional.Ownership < 30 &
    RelValIndex >= 8
)
target_stocks[, c("Ticker", "Company", "RelValIndex", "Price", "Sector")]

hist_px <- read.csv("data/food_historical_prices.csv", stringsAsFactors = FALSE)
hist_px$Date <- as.Date(hist_px$Date)

sym <- if ("GIS" %in% hist_px$Symbol) "GIS" else unique(hist_px$Symbol)[1]
one <- subset(hist_px, Symbol == sym)
one <- one[order(one$Date), ]
one$MovAvg50 <- NA
one$MovAvg200 <- NA
if (nrow(one) >= 50) one$MovAvg50[50:nrow(one)] <- rollmean(one$AdjClose, 50, align = "right")
if (nrow(one) >= 200) one$MovAvg200[200:nrow(one)] <- rollmean(one$AdjClose, 200, align = "right")
pc <- melt(one[, c("Date", "AdjClose", "MovAvg50", "MovAvg200")], id = "Date")
ggplot(pc, aes(Date, value, color = variable)) +
  geom_line() +
  ggtitle(paste(sym, "AdjClose + 50/200-day MAs")) +
  ylab("Price")

ggplot(hist_px, aes(Date, AdjClose, color = Symbol)) +
  geom_line() +
  ggtitle("Target food names — daily AdjClose")

price_summaries <- ddply(hist_px, "Symbol", summarise,
                         open = Open[1],
                         high = max(High, na.rm = TRUE),
                         low = min(Low, na.rm = TRUE),
                         close = AdjClose[length(AdjClose)])
summary_long <- melt(price_summaries, id = "Symbol")
ggplot(summary_long, aes(variable, value, fill = Symbol)) +
  geom_bar(stat = "identity", position = "dodge") +
  facet_wrap(~ Symbol, scales = "free_y") +
  ggtitle("Open / High / Low / Close (window)") +
  theme(legend.position = "none")

## Alternate code


In [ ]:
if (requireNamespace("dplyr", quietly = TRUE)) {
  sector_dplyr <- dplyr::summarise(dplyr::group_by(foodviz, Sector),
                                   Sector_Avg_Price = mean(Price, na.rm = TRUE))
  print(head(sector_dplyr))
}

sector_ddply <- ddply(foodviz, "Sector", summarise, Price = mean(Price, na.rm = TRUE))
print(sector_ddply)

k <- 50
w <- rep(1 / k, k)
ma_base <- as.numeric(stats::filter(one$AdjClose, w, sides = 1))
tail(na.omit(ma_base), 1)
tail(na.omit(one$MovAvg50), 1)

## More practice


In [ ]:
# 1. Median-based index
med_sector <- aggregate(cbind(Price, P.E, PEG, P.S, P.B) ~ Sector, data = foodviz, FUN = median)
names(med_sector)[-1] <- paste0("MS_", names(med_sector)[-1])
tmp <- merge(foodviz, med_sector, by = "Sector")
tmp$MedIdx <- (tmp$P.E < tmp$MS_P.E) + (tmp$PEG < tmp$MS_PEG) +
  (tmp$P.S < tmp$MS_P.S) + (tmp$P.B < tmp$MS_P.B) + (tmp$Price < tmp$MS_Price)
c(mean_ge8 = sum(foodviz$RelValIndex >= 8, na.rm = TRUE),
  median_ge4 = sum(tmp$MedIdx >= 4, na.rm = TRUE))

# 2. Food-quality extras
gm_med <- median(foodviz$Gross.Margin, na.rm = TRUE)
foodviz$QGM <- as.integer(foodviz$Gross.Margin > gm_med)
foodviz$QFC <- as.integer(foodviz$Food.Cost.. < 40)
foodviz$RelValPlus <- foodviz$RelValIndex + foodviz$QGM + foodviz$QFC
table(foodviz$RelValPlus)

# 3. Return correlations
wide <- dcast(hist_px, Date ~ Symbol, value.var = "AdjClose")
rets <- as.data.frame(lapply(wide[-1], function(x) diff(log(x))))
print(round(cor(rets, use = "pairwise.complete.obs"), 2))

# 4. Gross-margin boxes
gm_box <- subset(foodviz, Sector %in% c("Restaurants", "Grocery Retail"))
ggplot(gm_box, aes(Sector, Gross.Margin, fill = Sector)) +
  geom_boxplot(outlier.alpha = 0.3) +
  ggtitle("Gross margin: Restaurants vs Grocery Retail")

# 5. Restaurant SSS
if ("SSS.." %in% names(foodviz)) {
  rest <- subset(foodviz, Sector == "Restaurants" & !is.na(SSS..))
  print(head(rest[order(-rest$SSS..), c("Ticker", "Company", "SSS..")], 8))
}

## Simulation / what-if


In [ ]:
idx_cut <- 8
price_lo <- 20
price_hi <- 100
max_de <- 1
max_beta <- 1.5
max_inst <- 30
usa_only <- TRUE
max_foodcost <- 80
noise_sd <- 0
set.seed(21)

sim <- foodviz
if (noise_sd > 0) sim$Price <- sim$Price + rnorm(nrow(sim), 0, noise_sd)
keep_usa <- if (usa_only) sim$Country == "USA" else TRUE
passers <- subset(
  sim,
  keep_usa &
    Price > price_lo & Price < price_hi &
    Volume > 10000 &
    EPS..ttm. > 0 &
    EPS.growth.next.year > 0 &
    EPS.growth.next.5.years > 0 &
    Total.Debt.Equity < max_de &
    Beta < max_beta &
    Institutional.Ownership < max_inst &
    Food.Cost.. < max_foodcost &
    RelValIndex >= idx_cut
)
n_pass <- nrow(passers)
n_pass
if (n_pass) {
  print(median(passers$Price, na.rm = TRUE))
  print(sort(table(passers$Sector), decreasing = TRUE))
}

cuts <- 4:10
n_by_cut <- sapply(cuts, function(k) sum(sim$RelValIndex >= k, na.rm = TRUE))
sweep <- data.frame(idx_cut = cuts, n = n_by_cut)
ggplot(sweep, aes(idx_cut, n)) +
  geom_line() + geom_point() +
  ggtitle("Food names with RelValIndex ≥ cutoff") +
  xlab("cutoff") + ylab("count")

## Audience rewrite

**Expert / CPG quant.** RelValIndex is an unweighted count of ten “below cross-sectional mean” events on Price, P/E, PEG, P/S and P/B at food-sector and food-industry grain. Means are not robust — GODIVA at $18,750 is the worked example — and the industry merge drops names with incomplete comps. Flags ignore gap size (1 bp = 40%). Cocoa, cattle, and freight are missing factors. Historical MA overlays are in-sample. Do not read RelValIndex ≥ 8 as expected residual return versus a staples benchmark.

**Technician / category screener.** Pipeline: `read.csv("data/foodviz.csv")` → `clean_numeric` on columns 7+ → `make.names` → drop `Ticker == "GODIVA"` → melt/dcast sector and industry means → ten `< avg` flags → `rowSums` → subset USA, Price 20–100, Volume > 10k, EPS and both growth fields > 0, D/E < 1, Beta < 1.5, InstOwn < 30, index ≥ 8. Extra knobs: `max_foodcost`, `Gross.Margin`. Histories: `data/food_historical_prices.csv`. If the vendor adds a column, stop using hard indexes.

**Executive / CPG CFO or IC.** One Swiss luxury-chocolate listing made Confectionery look “expensive” on a mean-price bar. After removing it, food-sector average prices sit in a narrow band. A neighbor-comparison score plus a conservative quality screen leaves a short US CPG/restaurant list — a conversation starter for the coverage universe, not a buy list and not a slotting decision. Moving-average charts show path and volatility through the 2011–12 input-cost window; they do not forecast next quarter’s cereal volumes.

**Nonspecialist / grocery shopper.** We compared each food company’s share price to other companies in the same aisle (cereal vs cereal, burgers vs burgers), threw out one extreme luxury-chocolate price that was warping the average, and kept a short US list in a mid price range with manageable debt. Then we drew how those prices moved over a few years. That is a screening story, not a coupon and not a recommendation to buy stock — or chocolate.


## Key takeaways

- A food snapshot is still a *decorated* CSV: `%` `$` `,` must die before `mean()`.
- Luxury / Class-A / ADR names warp aisle-level averages the same way BRK-A warped Financials.
- Relative value is a peer comparison inside Packaged Foods / Restaurants / Protein — not a DCF of a cocoa contract.
- Food Cost % and Gross Margin are the extra levers this adaptation adds; they belong in practice and simulation, not as silent substitutes for P/E.
- Screens shrink the list. MAs describe the survivors. Neither is a procurement award.
- Rewrite the finding for quant, category manager, CFO, and shopper before it leaves the lab.
